<a href="https://colab.research.google.com/github/brgael/GTD2401-ACD-GA-AD26-07-ml-intermedio/blob/main/03.%20VariablesCategoricas.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Variables Categóricas

En este JNB se revisará el tema de Variables Categóricas.

In [2]:
# Read the data
import pandas as pd
from sklearn.model_selection import train_test_split

url = 'https://raw.githubusercontent.com/humbertgaar/ml1-unisabana-datasets/main/melb_data.csv'
data = pd.read_csv(url)

# Separate target from predictors
y = data.Price
X = data.drop(['Price'], axis=1)

# To keep things simple, we'll drop columns with missing values
cols_with_missing = [col for col in X.columns if X[col].isnull().any()]
X = X.drop(cols_with_missing, axis=1)

# Break off validation set
X_train_full, X_valid_full, y_train, y_valid = train_test_split(
    X, y, train_size=0.8, test_size=0.2, random_state=0
)

# "Cardinality" means the number of unique values in a column
# Select categorical columns with relatively low cardinality
low_cardinality_cols = [
    cname for cname in X_train_full.columns
    if X_train_full[cname].nunique() < 10
    and X_train_full[cname].dtype == "object"
]

# Select numerical columns
numerical_cols = [
    cname for cname in X_train_full.columns
    if X_train_full[cname].dtype in ['int64', 'float64']
]

# Keep selected columns only
my_cols = low_cardinality_cols + numerical_cols

X_train = X_train_full[my_cols].copy()
X_valid = X_valid_full[my_cols].copy()

print("Datos cargados correctamente")
print("Tamaño de X_train:", X_train.shape)
print("Tamaño de X_valid:", X_valid.shape)

Datos cargados correctamente
Tamaño de X_train: (10864, 12)
Tamaño de X_valid: (2716, 12)


In [3]:
# Get list of categorical variables
s = (X_train.dtypes == 'object')
object_cols = list(s[s].index)

print("Categorical variables:")
print(object_cols)

Categorical variables:
['Type', 'Method', 'Regionname']


In [4]:
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error

def score_dataset(X_train, X_valid, y_train, y_valid):
    model = RandomForestRegressor(n_estimators=100, random_state=0)
    model.fit(X_train, y_train)
    preds = model.predict(X_valid)
    return mean_absolute_error(y_valid, preds)

In [5]:
# Drop categorical columns
drop_X_train = X_train.select_dtypes(exclude=['object'])
drop_X_valid = X_valid.select_dtypes(exclude=['object'])

print("MAE from Approach 1 (Drop categorical variables):")
print(score_dataset(drop_X_train, drop_X_valid, y_train, y_valid))

MAE from Approach 1 (Drop categorical variables):
175703.48185157913


In [6]:
from sklearn.preprocessing import OrdinalEncoder

# Make copy to avoid changing original data
label_X_train = X_train.copy()
label_X_valid = X_valid.copy()

# Apply ordinal encoder to each column with categorical data
ordinal_encoder = OrdinalEncoder(
    handle_unknown='use_encoded_value',
    unknown_value=-1
)

label_X_train[object_cols] = ordinal_encoder.fit_transform(
    X_train[object_cols]
)

label_X_valid[object_cols] = ordinal_encoder.transform(
    X_valid[object_cols]
)

print("MAE from Approach 2 (Ordinal Encoding):")
print(score_dataset(label_X_train, label_X_valid, y_train, y_valid))

MAE from Approach 2 (Ordinal Encoding):
165936.40548390493


In [7]:
from sklearn.preprocessing import OneHotEncoder

# Apply one-hot encoder to each column with categorical data
try:
    OH_encoder = OneHotEncoder(handle_unknown='ignore', sparse_output=False)
except TypeError:
    OH_encoder = OneHotEncoder(handle_unknown='ignore', sparse=False)

OH_cols_train = pd.DataFrame(
    OH_encoder.fit_transform(X_train[object_cols])
)

OH_cols_valid = pd.DataFrame(
    OH_encoder.transform(X_valid[object_cols])
)

# One-hot encoding removed index; put it back
OH_cols_train.index = X_train.index
OH_cols_valid.index = X_valid.index

# Remove categorical columns (will replace with one-hot encoding)
num_X_train = X_train.drop(object_cols, axis=1)
num_X_valid = X_valid.drop(object_cols, axis=1)

# Add one-hot encoded columns to numerical features
OH_X_train = pd.concat([num_X_train, OH_cols_train], axis=1)
OH_X_valid = pd.concat([num_X_valid, OH_cols_valid], axis=1)

# Ensure all columns have string type
OH_X_train.columns = OH_X_train.columns.astype(str)
OH_X_valid.columns = OH_X_valid.columns.astype(str)

print("MAE from Approach 3 (One-Hot Encoding):")
print(score_dataset(OH_X_train, OH_X_valid, y_train, y_valid))

MAE from Approach 3 (One-Hot Encoding):
166089.4893009678


In [8]:
# Compare the three approaches

results = pd.DataFrame({
    'Approach': [
        'Drop categorical variables',
        'Ordinal Encoding',
        'One-Hot Encoding'
    ],
    'MAE': [
        score_dataset(drop_X_train, drop_X_valid, y_train, y_valid),
        score_dataset(label_X_train, label_X_valid, y_train, y_valid),
        score_dataset(OH_X_train, OH_X_valid, y_train, y_valid)
    ]
})

print(results.sort_values(by='MAE'))

                     Approach            MAE
1            Ordinal Encoding  165936.405484
2            One-Hot Encoding  166089.489301
0  Drop categorical variables  175703.481852
